# Bonus case: FlyArystan passenger-claim verification bottleneck

This is an original bottleneck, not the turnaround example in the brief. Public sources show refund restrictions, written-claim submission issues, third-party refund routing, passenger complaints, and court-enforced correction of passenger-rights rules. The internal verification mechanism is not publicly disclosed; this notebook tests a transparent simulated mechanism rather than claiming internal access.

Sources: [Kazakhstan Ministry of Trade and Integration](https://www.gov.kz/org/mti/press/news/details/1334367?lang=en), [FlyArystan official delay policy](https://flyarystan.com/en/cancellation-policy/), [Consumer Protection Committee](https://www.gov.kz/org/mti-kzpp/press/news/details/1149776?lang=ru), [Tengrinews passenger refund report](https://en.tengrinews.kz/my-country/flight-from-almaty-delayed-by-47-hours-passenger-reports-ticket-refund-issues-274994/).

## 1. Problem and bottleneck

Problem: after major disruptions, passengers face refunds, compensation, and complaint escalation. Bottleneck: the post-disruption claim queue—intake, eligibility verification, and routing to the original seller or payment channel. This is distinct from aircraft turnaround. The dataset contains no names, IINs, booking numbers, phone numbers, or addresses.

## 2. Cleaning

`data_raw.csv` has 1,204 rows: 1,200 simulated claims plus four exact duplicates and mixed date/unit/currency formats. The deterministic sequence drops duplicate `claim_id` values, parses dates and numeric fields, trims categories, and validates the final 1,200-row clean dataset.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency
raw = pd.read_csv('data_raw.csv')
raw.shape


(1204, 12)

In [2]:
df=raw.copy()
df['claim_id']=df['claim_id'].astype('string').str.strip()
df=df.drop_duplicates(subset='claim_id',keep='first').copy()
df['claim_date']=pd.to_datetime(df['claim_date'],format='mixed',dayfirst=True,errors='coerce')
for c in ['flight_delay_hours','claim_value_kzt','required_document_count','verification_cycle_mins','claim_denied_flag']:
    df[c]=pd.to_numeric(df[c].astype('string').str.replace(',','',regex=False).str.extract(r'(-?\d+(?:\.\d+)?)')[0],errors='coerce')
for c in ['disruption_event_id','purchase_channel','route','verification_mode','multi_flight_shock_flag']:
    df[c]=df[c].astype('string').str.strip()
df=df.sort_values('claim_id').reset_index(drop=True)
assert df.shape==(1200,12)
assert df.claim_id.is_unique and df.isna().sum().sum()==0
df.head()


,claim_id,claim_date,disruption_event_id,purchase_channel,route,flight_delay_hours,claim_value_kzt,verification_mode,required_document_count,verification_cycle_mins,multi_flight_shock_flag,claim_denied_flag
0,CL-00001,2026-03-11,EVT-0125,Travel agency,ALA-UKK,5.2,52470,Manual,4,223.8,Routine,1
1,CL-00002,2026-03-11,EVT-0166,FlyArystan direct,ALA-NQZ,4.4,41015,Automated,1,95.7,Routine,0
2,CL-00003,2026-02-15,EVT-0041,Kaspi Travel,ALA-KZO,6.7,52785,Manual,4,255.3,Shock,0
3,CL-00004,2026-04-11,EVT-0022,Kaspi Travel,ALA-AKT,3.5,30167,Manual,2,205.5,Routine,0
4,CL-00005,2026-06-20,EVT-0088,FlyArystan direct,ALA-AKT,17.8,157788,Manual,3,194.0,Routine,0


## 3. Hypothesis pair 1: verification latency

H0_1: mean verification cycle time is identical for Manual and Automated verification. H1_1: Manual verification takes longer. We use an independent Welch two-sample t-test with α=0.05.

In [3]:
manual=df.loc[df.verification_mode=='Manual','verification_cycle_mins']
auto=df.loc[df.verification_mode=='Automated','verification_cycle_mins']
t_test=ttest_ind(manual,auto,equal_var=False,alternative='greater')
print({'manual_mean':manual.mean(),'automated_mean':auto.mean(),'difference':manual.mean()-auto.mean(),'t_statistic':t_test.statistic,'p_value':t_test.pvalue,'verdict':'H0_1 rejected' if t_test.pvalue<0.05 else 'H0_1 not rejected'})


{'manual_mean': np.float64(217.38732876712328), 'automated_mean': np.float64(98.00275974025973), 'difference': np.float64(119.38456902686355), 't_statistic': np.float64(45.06775026857587), 'p_value': np.float64(1.70673489969586e-260), 'verdict': 'H0_1 rejected'}


## 4. Hypothesis pair 2: disruption-shock denials

H0_2: claim denial is independent of whether a claim arrives during a multi-flight disruption shock. H1_2: denial is more common during a shock. We use a chi-square test of independence with α=0.05.

In [4]:
table=pd.crosstab(df.multi_flight_shock_flag,df.claim_denied_flag).reindex(index=['Routine','Shock'],columns=[0,1],fill_value=0)
chi2,p_value,dof,expected=chi2_contingency(table)
rates=df.groupby('multi_flight_shock_flag').claim_denied_flag.mean()
print(table)
print({'chi2_statistic':chi2,'p_value':p_value,'routine_denial_rate':rates['Routine'],'shock_denial_rate':rates['Shock'],'verdict':'H0_2 rejected' if p_value<0.05 else 'H0_2 not rejected'})


claim_denied_flag          0    1
multi_flight_shock_flag          
Routine                  751  141
Shock                    209   99
{'chi2_statistic': np.float64(37.17050186360724), 'p_value': np.float64(1.082387623539567e-09), 'routine_denial_rate': np.float64(0.15807174887892378), 'shock_denial_rate': np.float64(0.32142857142857145), 'verdict': 'H0_2 rejected'}


## 5. Limitations and advice

The simulated log does not include internal timestamps, bank settlement events, IIN validation, actual claim outcomes, passenger identifiers, or real queue volumes. Public evidence supports the customer-rights and refund-routing problem, not a disclosed internal system design. If H0 is rejected on live data, create a single claims queue, auto-check complete eligible claims, publish service-level targets, and measure time from submission to decision separately by purchase channel and disruption shock.